# Vision Transformer on MNIST dataset

MNIST (28x28 greyscale digits), patch size 7, d = 64.

| Step | Operation | Shape |
|-|-|-|
| Input | image | (B,1,28,28) |
| Patchify | 28/7 = 4 tiles per side -> 16 tiles of 7*7=49 numbers each | (B, 16, 49) |
| Linear projection | 49 -> 64 | (B, 16, 64) |
| Prepend [CLS] | 16 + 1 tokens | (B, 17, 64) |
| Add position embedding | learned table | (B, 17, 64) |
| Final LayerNorm, take token 0 | the [CLS] output | (B, 64) |
| Head | 64 -> 10 | (B, 10) |

For reference, the original ViT-Base on ImageNet is: 224×224×3 image, 16×16 patches, so 196 tiles of 16x16x3 = 768 numbers each, and d = 768, giving 197 tokens.

## Architecture

### Patch Embedding

- Input: (B, C=1, 28, 28)
- kernel_size = stride = 7
- d = 64

Input image has 1 channel (greyscale) -> 64 dimensions. That means the layer applies 64 distinct filters, transforming 7x7 patch into 64-dim vector.

Spartial dimensions:

$$
H_{\text{out}} = \left\lfloor \frac{H_{in} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1
$$

So the spartial dims is 4x4.

> Input (B, 1, 28, 28) -> (B, 64, 4, 4)

After that, it is flattened -> (B, 64, 16), then transposed -> (B, 16, 64) - the same layout that GPT uses.

The only difference from GPT is the missing `masked_fill`. Row i of att says “how much token i listens to each other token”.

### Encoder Block

dropout is omitted to keep the code minimal. The ViT paper uses it.

## Training

In [5]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn.functional as F

from vision_transformer.vit import ViT

In [6]:
model = ViT()

In [7]:
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.1307,), (0.3081,))])  # MNIST mean/std
train = DataLoader(datasets.MNIST("data", train=True,  download=True, transform=tf),
                   batch_size=128, shuffle=True)
test  = DataLoader(datasets.MNIST("data", train=False, download=True, transform=tf),
                   batch_size=512)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)

for epoch in range(3):
    model.train()
    for x, y in train:
        loss = F.cross_entropy(model(x), y)   # forward + loss
        opt.zero_grad()                       # clear old gradients
        loss.backward()                       # backprop
        opt.step()                            # update
    model.eval()
    correct = 0
    with torch.no_grad():                     # inference: no graph needed
        for x, y in test:
            correct += (model(x).argmax(dim=1) == y).sum().item()
    print(f"epoch {epoch+1}: test acc {correct / len(test.dataset):.4f}")

epoch 1: test acc 0.9254
epoch 2: test acc 0.9591
epoch 3: test acc 0.9665


This is small, to see ViT's data hunger, swap to CIFAR-10 (`img=32, patch=4, in_ch=3`).